## Импорт библиотек

In [1]:
import pandas as pd
import sqlite3

## Подключение к базе

In [2]:
connection = sqlite3.connect("../data/checking-logs.sqlite")

## Обработка и анализ данных

In [3]:
pd.io.sql.read_sql(
    """
    SELECT
        uid,
        labname,
        first_commit_ts,
        first_view_ts,
        CASE
            WHEN first_commit_ts < first_view_ts THEN 'before'
            ELSE 'after'
        END AS time
    FROM test
    WHERE labname != 'project1';
    """,
    connection
)

Out[3]: 
        uid  labname  ...               first_view_ts    time
0    user_1   laba04  ...  2020-04-26 21:53:59.624136  before
1    user_1  laba04s  ...  2020-04-26 21:53:59.624136  before
2    user_1   laba05  ...  2020-04-26 21:53:59.624136   after
3    user_1   laba06  ...  2020-04-26 21:53:59.624136   after
4    user_1  laba06s  ...  2020-04-26 21:53:59.624136   after
5   user_10   laba04  ...  2020-04-18 12:19:50.182714   after
6   user_10  laba04s  ...  2020-04-18 12:19:50.182714   after
7   user_10   laba05  ...  2020-04-18 12:19:50.182714   after
8   user_10   laba06  ...  2020-04-18 12:19:50.182714   after
9   user_10  laba06s  ...  2020-04-18 12:19:50.182714   after
10  user_14   laba04  ...  2020-04-18 10:53:52.623447   after
11  user_14  laba04s  ...  2020-04-18 10:53:52.623447   after
12  user_14   laba05  ...  2020-04-18 10:53:52.623447   after
13  user_17   laba04  ...  2020-04-18 10:56:55.833899   after
14  user_17  laba04s  ...  2020-04-18 10:56:55.833899   after

## Обработка и анализ данных

In [4]:
pd.io.sql.read_sql(
    """
    SELECT uid
    FROM (
        SELECT
        uid,
        labname,
        first_commit_ts,
        first_view_ts,
        CASE
        WHEN first_commit_ts < first_view_ts THEN 'before'
              ELSE 'after'
        END AS time
        FROM test
        WHERE labname != 'project1'
    ) AS labeled
    GROUP BY uid
    HAVING COUNT(DISTINCT time) = 2
    """,
    connection
)

Out[4]: 
       uid
0   user_1
1  user_18
2  user_19
3  user_21
4  user_25
5  user_28
6   user_3


## Обработка и анализ данных

In [5]:
pd.io.sql.read_sql(
    """
    SELECT
        uid,
        labname,
        first_commit_ts,
        first_view_ts,
        CASE
            WHEN first_commit_ts < first_view_ts THEN 'before'
            ELSE 'after'
        END AS time, 
        (strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0 AS diff
    FROM test AS t
    JOIN deadlines AS d ON t.labname = d.labs
    WHERE labname != 'project1';
    """,
    connection
)

Out[5]: 
        uid  labname  ...    time        diff
0    user_1   laba04  ...  before   -6.894722
1    user_1  laba04s  ...  before   -6.796667
2    user_1   laba05  ...   after  -28.744722
3    user_1   laba06  ...   after -175.556667
4    user_1  laba06s  ...   after -107.606111
5   user_10   laba04  ...   after  -39.585278
6   user_10  laba04s  ...   after  -39.368056
7   user_10   laba05  ...   after  -52.542500
8   user_10   laba06  ...   after -132.341944
9   user_10  laba06s  ...   after -112.374444
10  user_14   laba04  ...   after -200.766389
11  user_14  laba04s  ...   after -193.491389
12  user_14   laba05  ...   after  -84.448611
13  user_17   laba04  ...   after  -81.591667
14  user_17  laba04s  ...   after  -35.817500
15  user_17   laba05  ...   after  -34.643056
16  user_17   laba06  ...   after  -80.641111
17  user_17  laba06s  ...   after  -78.345000
18  user_18   laba04  ...  before   -4.196667
19  user_18  laba04s  ...  before   -3.934167
20  user_18   laba05  ...

## Обработка и анализ данных

In [6]:
test_results = pd.io.sql.read_sql(
    """
    SELECT time, AVG(diff) AS avg_diff
    FROM (
        SELECT
            t.uid,
            CASE
                WHEN t.first_commit_ts < t.first_view_ts
                    THEN 'before'
                ELSE 'after'
            END AS time,
            (
                strftime('%s', t.first_commit_ts) - d.deadlines
            ) / 3600.0 AS diff
        FROM test AS t
        JOIN deadlines AS d ON t.labname = d.labs
        WHERE t.labname != 'project1'
    ) AS labeled
    WHERE uid IN (
        SELECT uid
        FROM (
            SELECT
                uid,
                CASE
                    WHEN first_commit_ts < first_view_ts
                        THEN 'before'
                    ELSE 'after'
                END AS time
            FROM test
            WHERE labname != 'project1'
        ) AS periods
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2
    )
    GROUP BY time;
    """,
    connection
)

test_results

Out[6]: 
     time    avg_diff
0   after -105.229241
1  before  -61.156632


## Обработка и анализ данных

In [7]:
control_results = pd.io.sql.read_sql(
    """
    SELECT time, AVG(diff) AS avg_diff
    FROM (
        SELECT
            t.uid,
            CASE
                WHEN t.first_commit_ts < t.first_view_ts
                    THEN 'before'
                ELSE 'after'
            END AS time,
            (
                strftime('%s', t.first_commit_ts) - d.deadlines
            ) / 3600.0 AS diff
        FROM control AS t
        JOIN deadlines AS d ON t.labname = d.labs
        WHERE t.labname != 'project1'
    ) AS labeled
    WHERE uid IN (
        SELECT uid
        FROM (
            SELECT
                uid,
                CASE
                    WHEN first_commit_ts < first_view_ts
                        THEN 'before'
                    ELSE 'after'
                END AS time
            FROM control
            WHERE labname != 'project1'
        ) AS periods
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2
    )
    GROUP BY time;
    """,
    connection
)

control_results

Out[7]: 
     time    avg_diff
0   after -110.985643
1  before -131.340608


В группе test после первого посещения ленты средняя дельта уменьшилась с −61,16 до −105,23 часа: первые отправки происходили раньше относительно дедлайна. В контрольной группе средняя дельта увеличилась с −131,34 до −110,99 часа. Наблюдаемые изменения согласуются с гипотезой о положительном влиянии ленты, однако не позволяют считать её доказанной без дополнительного статистического анализа.

## Закрытие соединения

In [8]:
connection.close()